In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 3.8 Maxwell's Equations and Electromagnetic Waves

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume III — Classical Electrodynamics",
    number="3.8",
    title="Maxwell's Equations and Electromagnetic Waves",
    blurb="The synthesis: one term added for consistency completes the four "
    "equations, they combine into a wave equation, and its speed — computed from "
    "the electric and magnetic constants alone — turns out to be the speed of light.",
    difficulty="advanced",
    estimate="130–160 min",
)

## Notebook overview

This is the climax of the volume. Across the last seven notebooks we built the field
equations one law at a time: Gauss for $\mathbf E$ ([§3.3](gauss-law.ipynb)), no
monopoles for $\mathbf B$
([§3.6](magnetostatics.ipynb)), Ampère for steady currents
([§3.6](magnetostatics.ipynb)), and Faraday's induction ([§3.7](induction.ipynb)).
Three of the
four were complete; one, Ampère's law, was quietly broken the moment fields varied in
time. Maxwell found the flaw, fixed it with a single new term, and in doing so turned a
collection of empirical laws into one dynamical theory whose solutions travel, at a
speed that turns out to be the speed of light.

We follow that path. First the inconsistency: $\nabla\times\mathbf B=\mu_0\mathbf J$
cannot survive a changing charge density, and the charging capacitor shows it
concretely, with $\mathbf B$ circulating around a gap that no conduction current
crosses. Maxwell's fix is the **displacement current** $\mu_0\varepsilon_0\,\partial_t
\mathbf E$, the exact mirror of Faraday's term: where a changing $\mathbf B$ makes an
$\mathbf E$ ([§3.7](induction.ipynb)), a changing $\mathbf E$ makes a $\mathbf B$.
That symmetry is what
lets a disturbance sustain itself.

With the equations complete we take the curl of Faraday's law, substitute
Ampère–Maxwell, and watch a **wave equation** fall out, with speed
$v=1/\sqrt{\mu_0\varepsilon_0}$. Computing that number from the electric and magnetic
constants alone, with no optics anywhere in the inputs, gives $2.998\times10^8\,$m/s:
**light is an electromagnetic wave**. We then build an explicit plane wave, confirm it
satisfies the wave equation numerically, animate it (the one place a moving picture is
truly warranted, the iconic image of the subject), and read off its structure:
$\mathbf E\perp\mathbf B\perp\mathbf k$, in phase, with $|\mathbf E|=c|\mathbf B|$. The
Poynting vector $\mathbf S=\tfrac1{\mu_0}\mathbf E\times\mathbf B$ tells us the wave
carries energy. Finally the **Lorenz gauge** returns the gauge freedom of
[§3.6](magnetostatics.ipynb) as a
*tool*: the right gauge choice decouples the potentials into two clean wave equations.

Everything is in **SI units**, with $\mu_0=4\pi\times10^{-7}\,$T·m/A and
$\varepsilon_0=8.854\times10^{-12}\,$F/m. A propagating wave is genuine motion, so one
figure here is animated; everything else is a still.

> **How to read the checks.** Each exercise ends with a `validate` call against an
> independent fact: a displacement current equal to the conduction current, the four
> equations holding on a test field, $1/\sqrt{\mu_0\varepsilon_0}$ equal to $c$, a
> plane wave satisfying $\nabla^2 E=\mu_0\varepsilon_0\,\partial_t^2 E$, the intensity
> $E_0^2/2\mu_0 c$. A ✓ is strong evidence; a ✗ is a prompt to *locate the
> discrepancy*, not a verdict.
>
> **A numerical-differentiation note.** Several checks take first or second
> derivatives of a sampled field with `numpy.gradient`, whose one-sided stencils at
> the array edges are less accurate than the centred interior stencil; we therefore
> validate on the **interior** (excluding the end points), the array-edge subtlety met
> since [§3.6](magnetostatics.ipynb).

> **Scope.** A working review, not a full course. See Nolting, *Theoretical Physics 3*
> {cite}`nolting3`; Griffiths, *Introduction to Electrodynamics* {cite}`griffiths_em`
> (ch. 7, 9); Jackson {cite}`jackson` (ch. 6–7).

## Theory in brief

### The inconsistency Maxwell found

Take the divergence of the steady Ampère law $\nabla\times\mathbf B=\mu_0\mathbf J$.
The left side vanishes identically (the divergence of a curl is zero), so it demands

```{math}
:label: eq-inconsistency
\nabla\cdot\mathbf J = 0,
```

but charge conservation says $\nabla\cdot\mathbf J=-\partial_t\rho$, which is nonzero
wherever charge piles up. The two collide at a charging capacitor: current flows into
the plates, $\mathbf B$ circulates around the wire, yet **no conduction current crosses
the gap**. Ampère's law, applied to a surface bulging through the gap, gives the wrong
answer.

### The displacement current

Maxwell's fix is to add a term built from the changing field between the plates,

```{math}
:label: eq-displacement
\nabla\times\mathbf B = \mu_0\mathbf J + \mu_0\varepsilon_0\frac{\partial\mathbf E}{\partial t}.
```

The new piece $\mu_0\varepsilon_0\,\partial_t\mathbf E$ is the **displacement
current**: a *changing electric field* sources $\mathbf B$, the exact mirror of
Faraday's *changing magnetic field* sourcing $\mathbf E$. In the capacitor gap
$\varepsilon_0\,\partial_t\mathbf E$ carries precisely the conduction current that was
missing, and taking the divergence of {eq}`eq-displacement` now reproduces charge
conservation exactly.

### Maxwell's equations, complete

Collected, the four equations are

```{math}
:label: eq-maxwell
\nabla\cdot\mathbf E = \frac{\rho}{\varepsilon_0}, \quad
\nabla\cdot\mathbf B = 0, \quad
\nabla\times\mathbf E = -\frac{\partial\mathbf B}{\partial t}, \quad
\nabla\times\mathbf B = \mu_0\mathbf J + \mu_0\varepsilon_0\frac{\partial\mathbf E}{\partial t}.
```

The dynamical symmetry of the two curl equations, a changing $\mathbf B$ making
$\mathbf E$ and a changing $\mathbf E$ making $\mathbf B$, is the feedback that permits
a self-sustaining disturbance.

### The wave equation and the speed of light

In vacuum ($\rho=0$, $\mathbf J=0$) take the curl of Faraday's law and use the identity
$\nabla\times(\nabla\times\mathbf E)=\nabla(\nabla\cdot\mathbf E)-\nabla^2\mathbf E$.
The first term vanishes ($\nabla\cdot\mathbf E=0$), and substituting Ampère–Maxwell for
$\nabla\times\mathbf B$ leaves

```{math}
:label: eq-wave
\nabla^2\mathbf E = \mu_0\varepsilon_0\frac{\partial^2\mathbf E}{\partial t^2},
```

a wave equation (and the identical one for $\mathbf B$) with speed
$v=1/\sqrt{\mu_0\varepsilon_0}$. Evaluated from the constants,

```{math}
:label: eq-speed-of-light
\frac{1}{\sqrt{\mu_0\varepsilon_0}} = 2.998\times10^8\,\mathrm{m/s} = c .
```

Computed from electricity and magnetism alone, out comes the speed of light:
**light is an electromagnetic wave**.

### Plane waves and energy flux

The simplest solutions are **plane waves**,

```{math}
:label: eq-plane-wave
\mathbf E = E_0\cos(kx-\omega t)\,\hat{\mathbf y}, \quad
\mathbf B = \frac{E_0}{c}\cos(kx-\omega t)\,\hat{\mathbf z}, \quad \omega = ck,
```

with $\mathbf E\perp\mathbf B\perp\mathbf k$ (transverse), oscillating in phase and
obeying $|\mathbf E|=c|\mathbf B|$; the direction of $\mathbf E$ is the polarization.
The energy flux is the **Poynting vector**, the flux term of the local
energy-conservation law that follows from Maxwell's equations and the Lorentz force
(Poynting's theorem; Jackson {cite}`jackson`, ch. 6, carries the derivation out in
full),

```{math}
:label: eq-poynting
\mathbf S = \frac{1}{\mu_0}\,\mathbf E\times\mathbf B,
```

whose time average for a plane wave is the intensity $\langle S\rangle=E_0^2/2\mu_0 c$.
Energy, and momentum, travel with the wave.

### The Lorenz gauge: gauge as a tool

Writing the fields with potentials, $\mathbf E=-\nabla V-\partial_t\mathbf A$ and
$\mathbf B=\nabla\times\mathbf A$ (the form forced by [§3.7](induction.ipynb)),
Maxwell's equations for $V$
and $\mathbf A$ come out coupled. Using the **gauge freedom** of
[§3.6](magnetostatics.ipynb) to impose the
**Lorenz condition**

```{math}
:label: eq-lorenz
\nabla\cdot\mathbf A + \frac{1}{c^2}\frac{\partial V}{\partial t} = 0
```

decouples them into two symmetric wave equations, $\Box V=-\rho/\varepsilon_0$ and
$\Box\mathbf A=-\mu_0\mathbf J$, with $\Box=\nabla^2-\tfrac1{c^2}\partial_t^2$ the
d'Alembertian. (Jackson {cite}`jackson`, ch. 6, carries the substitution and the
cancellation out in full.) The gauge freedom introduced in
[§3.6](magnetostatics.ipynb) is now a **tool**: the right choice
makes the wave structure manifest. (Forward:
[§3.12](relativistic-maxwell.ipynb) shows the Lorenz condition is
Lorentz-invariant, gauge as *structure*.)

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

from ecp import draw, validate
from ecp.animate import show

from scipy.constants import mu_0 as MU0  # vacuum permeability, T·m/A
from scipy.constants import epsilon_0 as EPS0  # vacuum permittivity, F/m
from scipy.constants import (
    c as C_MEAS,
)  # the measured speed of light, m/s (defined SI value)

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT


def second_deriv(f, coord, axis):
    """Centred finite-difference second derivative along one axis.

    The 3-point stencil (f[i+1] − 2f[i] + f[i−1])/h^2 on a uniform grid:
    physically the operator that turns a field into its local curvature, the
    heart of the wave equation. End indices wrap (`numpy.roll`) and are
    excluded by the interior slice used in validation.

    Parameters
    ----------
    f : numpy.ndarray
        Sampled field values.
    coord : numpy.ndarray
        Uniform 1-D coordinate array along ``axis``; its spacing sets h.
    axis : int
        Array axis along which to differentiate.

    Returns
    -------
    numpy.ndarray
        The second derivative, same shape as ``f``.
    """
    h = coord[1] - coord[0]
    return (np.roll(f, -1, axis=axis) - 2.0 * f + np.roll(f, 1, axis=axis)) / h**2

## Exercise 1 — The displacement current and the capacitor

The cleanest place to see Ampère's law fail is a **charging parallel-plate capacitor**.
A conduction current $I$ flows along the wire into the plates, and an Amperian loop
around the wire encloses it, so $\oint\mathbf B\cdot d\boldsymbol\ell=\mu_0 I$. But the
loop bounds many surfaces, and one can be drawn to bulge *between the plates*, where no
charge flows: through that surface the conduction current is zero
({numref}`fig-mw-capacitor-setup`). The steady Ampère law {eq}`eq-inconsistency` gives
two different answers for the same loop. Maxwell's displacement current
{eq}`eq-displacement` repairs it: the changing field in the gap,
$\varepsilon_0\,\partial_t\mathbf E$, carries exactly the missing current.

**This exercise (worked).** Take $I=1\,$A charging plates of area $A=0.01\,\mathrm{m}^2$.
The gap field is $E=Q/(\varepsilon_0 A)$ with $Q(t)=It$, so the displacement current is
$I_d=\varepsilon_0 A\,dE/dt$. Compute $dE/dt$ from a sampled $E(t)$ with
`numpy.gradient` and confirm $I_d$ equals the conduction current $I$, restoring
consistency.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    I_disp_interior,
    I_cond,
    "the displacement current ε₀A dE/dt equals the conduction current, making ∇×B consistent",
    rtol=1e-6,
)

## Exercise 2 — Maxwell's equations assembled

With the displacement current in hand the four equations {eq}`eq-maxwell` are complete:
Gauss ([§3.3](gauss-law.ipynb)), no monopoles ([§3.6](magnetostatics.ipynb)), Faraday
([§3.7](induction.ipynb)), and the now-corrected Ampère–Maxwell
law (Exercise 1). The decisive new feature is the **symmetry of the two curl
equations**: a changing $\mathbf B$ makes $\mathbf E$ and a changing $\mathbf E$ makes
$\mathbf B$, the mutual feedback that lets a disturbance propagate with no charges or
currents to sustain it.

**This exercise (worked synthesis).** Take the vacuum plane wave {eq}`eq-plane-wave`,
$\mathbf E=E_0\cos(kx-\omega t)\,\hat{\mathbf y}$ and $\mathbf B=(E_0/c)\cos(kx-\omega
t)\,\hat{\mathbf z}$, sampled on an $(x,t)$ grid, and verify all four equations hold
numerically (derivatives by `numpy.gradient`, checked on the **interior**): the two
divergences vanish identically, and the two curl relations
$\partial_x E_y=-\partial_t B_z$ (Faraday) and $-\partial_x B_z=\mu_0\varepsilon_0\,
\partial_t E_y$ (Ampère–Maxwell) hold. That Ampère–Maxwell holds on a source-free wave
is the cleanest test that the assembled set is self-consistent.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    bool(faraday_ok and ampere_ok and div_ok),
    "the assembled Maxwell equations hold on the vacuum plane wave (the two curl laws "
    "numerically, the two divergences identically)",
)

## Exercise 3 — The wave equation and the speed of light

Now the payoff. In vacuum, taking the curl of Faraday's law and using
$\nabla\times(\nabla\times\mathbf E)=\nabla(\nabla\cdot\mathbf E)-\nabla^2\mathbf E$
with $\nabla\cdot\mathbf E=0$ gives $-\nabla^2\mathbf E=-\partial_t(\nabla\times\mathbf
B)$; substituting Ampère–Maxwell $\nabla\times\mathbf B=\mu_0\varepsilon_0\,\partial_t
\mathbf E$ yields the wave equation {eq}`eq-wave`, $\nabla^2\mathbf E=\mu_0\varepsilon_0
\,\partial_t^2\mathbf E$, with speed $v=1/\sqrt{\mu_0\varepsilon_0}$.

**This exercise (worked).** That speed is built from two constants measured in entirely
non-optical experiments: $\varepsilon_0$ from the force between charges
([§3.1](coulomb-field.ipynb)) and
$\mu_0$ from the force between currents ([§3.6](magnetostatics.ipynb)). Compute
$1/\sqrt{\mu_0\varepsilon_0}$ and
compare it to the measured speed of light {eq}`eq-speed-of-light`. The agreement is the
whole argument: nothing about light went into the inputs, yet $c$ comes out. Light is an
electromagnetic wave, one of the great unifications in physics.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    c_predicted,
    C_MEAS,
    "1/√(μ₀ε₀) equals the speed of light — light is an electromagnetic wave",
    rtol=1e-6,
)

## Exercise 4 — A plane wave, and the wave it makes

Build the explicit plane wave {eq}`eq-plane-wave` and put it to the test. With
$E_0=1\,$V/m, wavelength $0.5\,$m (so $k=2\pi/\lambda$) and $\omega=ck$, the field
$E_y(x,t)=E_0\cos(kx-\omega t)$ should satisfy the wave equation {eq}`eq-wave` exactly.

**This exercise (worked).** On the $(x,t)$ grid of Exercise 2, form the second
derivatives $\partial_x^2 E_y$ and $\partial_t^2 E_y$ with a **centred
finite-difference** stencil (the `second_deriv` helper) and confirm $\partial_x^2 E_y=
\mu_0\varepsilon_0\,\partial_t^2 E_y$ on the **interior**, and that the dispersion
relation $\omega/k=c$ holds. The animation
({numref}`fig-mw-wave-anim`) is the iconic image of the subject: $\mathbf E$ and
$\mathbf B$ oscillating in mutually perpendicular planes while the whole pattern
marches forward at $c$.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    d2E_dx2[core],
    (MU0 * EPS0 * d2E_dt2)[core],
    "the plane wave satisfies the electromagnetic wave equation with v = c",
    rtol=1e-3,
)
validate.close(omega / k, c, "the dispersion relation is ω = ck", rtol=1e-6)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — The E, B, k relations

The plane wave {eq}`eq-plane-wave` packs three structural facts: the fields are
**transverse** ($\mathbf E\perp\mathbf k$ and $\mathbf B\perp\mathbf k$), **mutually
perpendicular** ($\mathbf E\perp\mathbf B$), and locked **in phase** with amplitudes
tied by $|\mathbf E|=c|\mathbf B|$. Together they say the wave is a pair of sinusoids
riding in orthogonal planes, the picture animated in Exercise 4.

**This exercise (worked).** With $\hat{\mathbf k}=\hat{\mathbf x}$,
$\mathbf E\parallel\hat{\mathbf y}$ and $\mathbf B\parallel\hat{\mathbf z}$:

1. Confirm the three perpendicularity dot products vanish.
2. **Derive** $B_z$ from $E_y$ via Faraday's law — integrate
   $\partial_t B_z=-\partial_x E_y$ in time
   (`scipy.integrate.cumulative_trapezoid`) — rather than restating the
   constructed field, and confirm the derived wave is in phase with
   $\mathbf E$ with amplitude $E_0/c$.
3. Confirm $E_0/B_0=c$ ({numref}`fig-mw-eb-snapshot`).

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(E0 / B0, c, "the fields satisfy |E| = c|B|", rtol=1e-6)
validate.check(perp < 1e-12, "E ⊥ B ⊥ k (transverse, mutually perpendicular)")
validate.close(
    derived_err,
    0.0,
    "B derived from E via Faraday's law is in phase with E and has amplitude E₀/c",
    atol=1e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 6 — The Poynting vector and intensity

A wave carries energy, and the **Poynting vector** {eq}`eq-poynting`
$\mathbf S=\tfrac1{\mu_0}\mathbf E\times\mathbf B$ measures the power per unit area it
transports. For the plane wave, $\mathbf E\times\mathbf B$ points along $+\hat{\mathbf
x}$ (the propagation direction) with magnitude $E_0 B_0\cos^2(kx-\omega
t)$, so the energy flows with the wave and pulses at twice its frequency.

**This exercise (student).**

1. Compute $S_x(t)=\tfrac1{\mu_0}E_0 B_0\cos^2(\omega t)$ at a fixed point
   over one period.
2. Time-average it by integrating with `numpy.trapezoid` and dividing by the
   period, and compare to the closed-form intensity
   $\langle S\rangle=E_0^2/(2\mu_0 c)$ ({numref}`fig-mw-poynting`).

Energy and momentum travelling with the wave are the basis of radiation
pressure and of how antennas radiate ([§3.10](radiation.ipynb)).

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    S_avg,
    S_closed,
    "the time-averaged intensity is E₀²/(2μ₀c)",
    rtol=1e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — The Lorenz gauge decouples the potentials

With the potentials $\mathbf E=-\nabla V-\partial_t\mathbf A$ and $\mathbf B=\nabla
\times\mathbf A$ (the form forced by [§3.7](induction.ipynb)), Maxwell's equations
become two coupled
equations for $V$ and $\mathbf A$, tangled by cross terms. The gauge freedom of
[§3.6](magnetostatics.ipynb),
$\mathbf A\to\mathbf A+\nabla\chi$, is exactly the slack needed to untangle them:
imposing the **Lorenz condition** {eq}`eq-lorenz` $\nabla\cdot\mathbf A+\tfrac1{c^2}
\partial_t V=0$ cancels the cross terms and leaves two symmetric wave equations,
$\Box V=-\rho/\varepsilon_0$ and $\Box\mathbf A=-\mu_0\mathbf J$, with $\Box=\nabla^2-
\tfrac1{c^2}\partial_t^2$.

**This exercise (worked; gauge arc stage 2).** In vacuum ($\rho=0$, $\mathbf J=0$) a
plane-wave vector potential $A_y=(E_0/\omega)\sin(kx-\omega t)$ (which gives back
$E_y=-\partial_t A_y$) must satisfy $\Box A_y=0$. Form the d'Alembertian with the
**centred finite-difference** second derivatives of the `second_deriv` helper and
confirm the residual vanishes on the **interior** (relative to the natural scale
$k^2|A_y|$). This is the gauge freedom of
[§3.6](magnetostatics.ipynb) turned into a **tool**: the right gauge makes the wave
structure fall straight out.
Forward, [§3.12](relativistic-maxwell.ipynb) shows the Lorenz condition is
Lorentz-invariant (gauge as *structure*),
and Vol VI makes the gauge physical (the Aharonov–Bohm effect).

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    box_resid,
    0.0,
    "in the Lorenz gauge a vacuum plane-wave potential satisfies □A = 0",
    atol=1e-2,
)

## Exercise 8 — The electromagnetic spectrum

Every electromagnetic wave shares the speed $c$, so frequency and wavelength trade off
through $\lambda=c/f$. That single relation spans the entire spectrum, from
kilometre-long radio waves to gamma rays smaller than a nucleus, all the same physics
at different scales.

**This exercise (student).** For representative frequencies across the named bands
(radio, microwave, infrared, visible, ultraviolet, X-ray, gamma), compute the
wavelength $\lambda=c/f$ and the photon energy $E=hf$ (with $h$ Planck's constant),
tabulate them, and confirm $\lambda f=c$ holds across the whole range
({numref}`fig-mw-spectrum`). The photon energy is a forward pointer: that light comes
in quanta $E=hf$ is where Vol VI begins.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    wavelengths * freqs,
    np.full_like(freqs, c),
    "all electromagnetic waves satisfy λf = c across the spectrum",
    rtol=1e-6,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 9 — What Maxwell unified

Stand back and take in what these four equations accomplished. Discovered piecemeal as
three apparently separate subjects, electricity (Gauss), magnetism (no monopoles,
Ampère) and induction (Faraday), they close, with one term added for consistency, into
a single dynamical theory. And that theory's free solutions are **waves that travel at
$c$**, which is the speed of light, so radio, microwaves, visible light, X-rays and
gamma rays are revealed as one phenomenon at different frequencies. Few equations in
physics have unified so much.

But the synthesis leaves a loose end that will not stay quiet. The wave equation
{eq}`eq-wave` picks out a definite speed $c=1/\sqrt{\mu_0\varepsilon_0}$, the same in
its derivation no matter who writes it down. A speed, but **relative to what?** Every
other wave we know travels at a fixed speed relative to a medium, and rides faster or
slower for an observer moving through that medium. Maxwell's equations name no medium
and no preferred frame. That single unanswered question, a universal speed with nothing
to measure it against, is the crack through which **special relativity** enters: the
capstone ([§3.12](relativistic-maxwell.ipynb)), which shows that $\mathbf E$ and
$\mathbf B$ are one object seen from
different frames, and that the Lorenz gauge condition met here is Lorentz-invariant.

**This exercise.** Make the closing tally concrete by collecting the earlier results:
the four equations are in hand, the displacement current switched on the dynamical
$\partial_t\mathbf E$ term (Exercise 1), and the speed they predict,
$1/\sqrt{\mu_0\varepsilon_0}$, equals $c$ (Exercise 3), the number that forces the
question of frames.

In [ ]:
# (solution hidden on the public site)


### Validation 9

In [ ]:
validate.check(
    bool(displacement_on) and bool(c_emerges),
    "the two measured pillars of the synthesis hold: I_d = I (Ex 1) and 1/√(μ₀ε₀) = c "
    "(Ex 3) — light, and the question of frames that forces relativity (§3.12)",
)

## Notebook summary

- **The displacement current** {eq}`eq-displacement`: the charging capacitor exposed
  the inconsistency {eq}`eq-inconsistency` in $\nabla\times\mathbf B=\mu_0\mathbf J$,
  and $\varepsilon_0\,\partial_t\mathbf E$ in the gap carries exactly the conduction
  current (Exercise 1).
- **Maxwell's equations complete** {eq}`eq-maxwell`: verified on a vacuum plane wave,
  the two curl laws numerically and the divergences identically (Exercise 2).
- **Light is an electromagnetic wave** {eq}`eq-wave`, {eq}`eq-speed-of-light`: the
  vacuum equations give $\nabla^2\mathbf E=\mu_0\varepsilon_0\,\partial_t^2\mathbf E$,
  and $1/\sqrt{\mu_0\varepsilon_0}=c$ from the constants alone (Exercise 3).
- **The plane wave** {eq}`eq-plane-wave`: it satisfies the wave equation to the
  finite-difference floor and has $\omega=ck$ (Exercise 4, the warranted animation);
  $\mathbf E\perp\mathbf B\perp\mathbf k$, in phase, with $|\mathbf E|=c|\mathbf B|$
  (Exercise 5).
- **Energy flux** {eq}`eq-poynting`: $\langle S\rangle=E_0^2/2\mu_0 c$ by time-averaging
  the Poynting vector (Exercise 6).
- **The Lorenz gauge** {eq}`eq-lorenz`: gauge freedom as a tool, decoupling the
  potentials into wave equations, with $\Box A=0$ for a vacuum wave (Exercise 7); the
  spectrum $\lambda f=c$ from radio to gamma (Exercise 8); and the unification, with the
  "speed relative to what?" question that opens onto relativity (Exercise 9).

## Outlook

- **Waves in media.** A refractive index $n$, dispersion ($n$ depending on frequency),
  and absorption all follow from how bound and free charges respond to the wave;
  reflection and refraction at an interface give the Fresnel relations.
- **Waveguides and cavities ([§3.9](waveguides-cavities.ipynb)).** Confining a wave
  with conducting boundaries
  quantises the modes it can carry, the electromagnetic analogue of a vibrating string.
- **Radiation ([§3.10](radiation.ipynb)).** The displacement current also explains
  how *accelerating
  charges launch* these waves; the Poynting flux met here becomes radiated power.
- **The covariant potentials ([§3.12](relativistic-maxwell.ipynb)).** The Lorenz-gauge
  wave equations $\Box V=-\rho/
  \varepsilon_0$, $\Box\mathbf A=-\mu_0\mathbf J$ are already relativistic in form;
  [§3.12](relativistic-maxwell.ipynb)
  makes that manifest, with gauge as *structure* and the field tensor $F^{\mu\nu}$.
- **The speed-relative-to-what puzzle ([§3.12](relativistic-maxwell.ipynb), Vol IV).**
  A universal $c$ with no medium
  is exactly what special relativity is built to explain; that is where the volume ends.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()